### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

c:\Users\lenovo\Downloads\Learning-Langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002AD381D9550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002AD381D9FD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The release year of the movie")
    rating: float = Field(description="The rating of the movie")

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002AD381D9550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002AD381D9FD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'The release year of the movie', 'type': 'integer'}, 'rating': {'description': 'The rating of the movie', 'type': 'number'}}, 'required': ['title', 'directo

In [6]:
model.invoke("What is the movie Inception about?")

AIMessage(content='<think>\nOkay, so I need to explain what the movie Inception is about. Let me start by recalling what I know. I remember it\'s directed by Christopher Nolan, and it\'s about dreams and something with layers. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who enters people\'s dreams to steal secrets. The movie has a lot of action, but there\'s also a complex plot involving different levels of dreams.\n\nFirst, I should outline the basic premise. Inception is about the idea of planting an idea into someone\'s mind instead of stealing it. That\'s called "inception" as opposed to "extraction." The protagonist, Cobb, is known for extraction, but he\'s offered a chance to have his criminal history erased if he can perform an inception. The target is a businessman named Robert Fischer. \n\nNow, the team that Cobb assembles includes various specialists. There\'s Arthur, who\'s the logistics expert, Ariadne the architect who designs the dream lands

In [9]:
response=model_with_structure.invoke("provide details about the movie Inception?")
response


Movie(title='Inception', director='Christopher Nolan', release_year=2010, rating=8.8)

Message output alongside parsed structure
- Mix of both structured and raw form.

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The release year of the movie")
    rating: float = Field(description="The rating of the movie")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("provide details about the movie Inception?")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what tools I have available. There\'s a Movie function that requires title, director, release year, and rating. I need to fill in those parameters. I know Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably high; IMDb gives it an 8.8. Let me make sure all required fields are included. Title is "Inception", director is "Christopher Nolan", release_year is 2010, and rating is 8.8. That should cover it. Let me structure the JSON accordingly.\n', 'tool_calls': [{'id': '7cw7tqhb8', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"release_year":2010,"title":"Inception"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 225, 'total_tokens': 408, 'completion_time': 0.329486959, 'completion_tokens_details': {'reasoning_to

#### Nested Structure

In [14]:
class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    role: str = Field(description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    cast: list[Actor] = Field(description="The main cast of the movie")
    genre: str = Field(description="The genre of the movie")
    budget: int = Field(description="The budget of the movie in millions")

model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("provide detailed information about the movie Inception?")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genre='Science Fiction, Action', budget=160)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

The only difference btw Pydantic and TypedDict is the output format: 
- Pydantic output --> Pydantic format
- TypedDict output --> Json/Dict format.

Does not provide runtime validation. ( If the LLM returns an integer where a string was specified, it will not raise an error, making it suitable for scenarios where strict data validation is not required. )

In [15]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [16]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [17]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
- A standard Python concept similar to TypedDict in its approach.

- Like TypedDict, it serves as a way to structure data but lacks the deep, automated field-level validation enforcement that Pydantic offers for LLM outputs.

In [32]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [33]:
class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [34]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')